# BirdNET Training Data Segmentation Script

This notebook prepares training and testing datasets for BirdNET from raw audio recordings.


In [7]:
# Imports
import os
import soundfile as sf
import numpy as np
from pathlib import Path
import sklearn
from sklearn.model_selection import train_test_split

In [2]:
# Parameters
input_dir = Path("/Volumes/theporp/SealRoarClips_ModelTraining")  # contains GS/, HS/, noise/
output_dir = Path("/Volumes/theporp/PNS_Birdnet")               # where train/test folders will go
segment_duration = 3.0                   # seconds
expected_sr = 48000                      # BirdNET default sample rate
test_size = 0.2                          # 20% for testing

classes = ["GS", "HS", "noise"]
output_dir.mkdir(exist_ok=True)

In [4]:
# Initialize counters
segment_counts = {"GS": 0, "HS": 0, "noise": 0}

def segment_audio(file_path, out_folder, label):
    y, sr = sf.read(file_path, always_2d=False)
    if sr != expected_sr:
        print(f"⚠️ WARNING: {file_path.name} has sample rate {sr} Hz (expected {expected_sr} Hz). "
              "Proceeding without resampling.")

    segment_samples = int(segment_duration * sr)
    total_samples = len(y)

    start = 0
    segment_index = 0

    while start < total_samples:
        end = min(start + segment_samples, total_samples)
        segment = y[start:end]

        # Pad if shorter than target length
        if len(segment) < segment_samples:
            pad_width = segment_samples - len(segment)
            segment = np.pad(segment, (0, pad_width), mode='constant')

        out_name = f"{label}_{file_path.stem}_{segment_index:04d}.wav"
        out_path = out_folder / label / out_name
        out_path.parent.mkdir(parents=True, exist_ok=True)
        sf.write(out_path, segment, sr)

        segment_index += 1
        segment_counts[label] += 1

        start += segment_samples

In [5]:
# Collect file paths, skipping hidden Apple metadata files
gs_files = [f for f in (input_dir / "GS").glob("*.wav") if not f.name.startswith("._")]
hs_files = [f for f in (input_dir / "HS").glob("*.wav") if not f.name.startswith("._")]
noise_files = [f for f in (input_dir / "noise").glob("*.wav") if not f.name.startswith("._")]


# Split GS and HS into train/test
gs_train, gs_test = train_test_split(gs_files, test_size=test_size, random_state=42)
hs_train, hs_test = train_test_split(hs_files, test_size=test_size, random_state=42)

In [8]:
# Create output folders
for split in ["train", "test"]:
    for label in classes:
        (output_dir / split / label).mkdir(parents=True, exist_ok=True)

# Process and save segments
# GS
for f in gs_train:
    segment_audio(f, output_dir / "train", "GS")
for f in gs_test:
    segment_audio(f, output_dir / "test", "GS")

# HS
for f in hs_train:
    segment_audio(f, output_dir / "train", "HS")
for f in hs_test:
    segment_audio(f, output_dir / "test", "HS")

# Noise (train only)
for f in noise_files:
    segment_audio(f, output_dir / "train", "noise")

# After processing all files
print("✅ Segmentation complete!")
for label, count in segment_counts.items():
    print(f"{label}: {count} segments created")

✅ Segmentation complete!
GS: 384 segments created
HS: 1312 segments created
noise: 0 segments created
